In [1]:
import os
import torch
from transformers import BertTokenizerFast, BertModel
from data import *
from train import *
from model import *
from evaluate import *

In [2]:
# Khởi tạo tokenizer và model BERT
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")
# Thêm token đặc biệt nếu chưa có
special_tokens = {"additional_special_tokens": ["[E1]", "[/E1]", "[E2]", "[/E2]"]}
num_added_toks = tokenizer.add_special_tokens(special_tokens)
bert.resize_token_embeddings(len(tokenizer))  # Resize embedding BERT

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(30526, 768, padding_idx=0)

# Load data

In [3]:
data_info = loadi2b2()

In [4]:
train_df = data_info['train_df']
valid_df = data_info['valid_df']
test_df = data_info['test_df']
num_classes = data_info['num_classes']
relation_map = data_info['label_mapping']
id2label = data_info['id2label']

type_map = {"UNKNOWN": 0, "OCCURRENCE": 1, "TREATMENT": 2, "TEST": 3, "DURATION": 4, "PROBLEM": 5, "CLINICAL_DEPT": 6, "EVIDENTIAL": 7,
            "DATE": 8, "FREQUENCY": 9, "TIME": 10,
        }
num_types = 11
rule_filename = 'rules_i2b2.txt'

cache_path = '/data/ddao/TRE/pretrained_models/Reasoning/i2b2/cache/'

os.makedirs(cache_path + "train/", exist_ok=True)
os.makedirs(cache_path + "test/", exist_ok=True)
os.makedirs(cache_path + "valid/", exist_ok=True)

train_token_cache_path = cache_path + "train/token_cache_bert.pt"
train_spacy_jsonl_path = cache_path + "train/spacy_cache_bert.jsonl"

test_token_cache_path = cache_path + "test/token_cache_bert.pt"
test_spacy_jsonl_path = cache_path + "test/spacy_cache_bert.jsonl"

valid_token_cache_path = cache_path + "valid/token_cache_bert.pt"
valid_spacy_jsonl_path = cache_path + "valid/spacy_cache_bert.jsonl"

batch_size = 128
window_size = 128
MAX_LENGTH = 256

In [5]:
train_loader = create_dataloader(train_token_cache_path, train_spacy_jsonl_path, batch_size=batch_size)
val_loader   = create_dataloader(valid_token_cache_path, valid_spacy_jsonl_path, batch_size=batch_size)
test_loader  = create_dataloader(test_token_cache_path, test_spacy_jsonl_path, batch_size=batch_size)

# Create Model

In [31]:
mode           = 'baseline_reasoning'
enc_layers     = 2
nhead          = 8
p_drop         = 0.1
step_size      = 1.0
lambda_kl      = 5.0
smooth_tau     = 0.1
lambda_entropy = 0.1
max_steps      = 50
epochs         = 60
lr             = 0.0001
lambda_distill = 5.0
lambda_energy  = 0.0
use_amp        = True
freeze_bert    = True
use_deq        = True
output_option  = 'direct_q' # 'direct_q' or 'refinement'
# save_path      = f"/data/ddao/TRE/pretrained_models/Reasoning/i2b2/{mode}_BERT_{output_option}.pt"
save_path      = f"/data/ddao/TRE/pretrained_models/Reasoning/i2b2/{mode}_BERT_{output_option}.pt"

In [32]:
model = create_model(
    mode=mode,
    bert=bert,
    num_classes=num_classes,
    num_types=num_types,
    type_map=type_map,
    relation_map=relation_map,
    enc_layers=enc_layers,
    nhead=nhead,
    p_drop=p_drop,
    step_size=step_size,
    lambda_kl=lambda_kl,
    smooth_tau=smooth_tau,
    lambda_entropy=lambda_entropy,
    max_steps=max_steps,
    rule_file_path=rule_filename,
    freeze_bert=freeze_bert,
    use_deq=use_deq,
    output_option=output_option
)

[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_BB
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_BO
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_OB
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_OO
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_AA
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_AO
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_OA
[UnifiedNeuralReasoningLayer] Hard Rule: R_Symm_BA
[UnifiedNeuralReasoningLayer] Hard Rule: R_Symm_AB
[UnifiedNeuralReasoningLayer] init: rules=9, rel=3, types=11, smooth_tau=0.1, lambda_kl=5.0, lambda_entropy=0.1


# Train

In [33]:
# train_model(model, train_loader, val_loader, epochs, lr, lambda_distill, save_path, use_amp=use_amp, lambda_energy=lambda_energy)

In [34]:
# train_baseline(model, train_loader, val_loader, epochs, lr, save_path, use_amp=use_amp)

In [35]:
train_baseline_reasoning(model, train_loader, val_loader, epochs, lr, lambda_distill, save_path, use_amp=use_amp, lambda_energy=lambda_energy)

Using device: cuda
Enforced learning rate: 0.0001


Evaluate (Baseline Reasoning): 100%|██████████| 40/40 [00:48<00:00,  1.21s/it]


Epoch 0: Loss=0.6335 (CE=0.6331, KL=0.0001, EQ=0.1050, EP=0.1679) | Val F1=0.7914
  Saved best model to /data/ddao/TRE/pretrained_models/Reasoning/i2b2/baseline_reasoning_BERT_direct_q.pt (F1: 0.7914)


Evaluate (Baseline Reasoning): 100%|██████████| 40/40 [00:48<00:00,  1.21s/it]


Epoch 1: Loss=0.5335 (CE=0.5331, KL=0.0001, EQ=0.1178, EP=0.1727) | Val F1=0.8065
  Saved best model to /data/ddao/TRE/pretrained_models/Reasoning/i2b2/baseline_reasoning_BERT_direct_q.pt (F1: 0.8065)


...

Epoch 57: Loss=0.0367 (CE=0.0366, KL=0.0000, EQ=0.1984, EP=0.2020) | Val F1=0.8683
  Saved best model to /data/ddao/TRE/pretrained_models/Reasoning/i2b2/baseline_reasoning_BERT_direct_q.pt (F1: 0.8683)


Evaluate (Baseline Reasoning): 100%|██████████| 40/40 [00:48<00:00,  1.20s/it]


Epoch 58: Loss=0.0360 (CE=0.0360, KL=0.0000, EQ=0.1990, EP=0.2022) | Val F1=0.8630


Evaluate (Baseline Reasoning): 100%|██████████| 40/40 [00:47<00:00,  1.20s/it]

Epoch 59: Loss=0.0340 (CE=0.0340, KL=0.0000, EQ=0.1992, EP=0.2022) | Val F1=0.8638
Training finished in 15696.94s.


# Evaluate

In [36]:
print("Load pretrained model:")
checkpoint   = torch.load(save_path)
model.load_state_dict(checkpoint['model_state_dict'])

Load pretrained model:


<All keys matched successfully>

In [37]:
cm, f1, p, r = evaluate_model(
    test_loader=test_loader,
    model=model
)
print('CM:\n', cm)
print('f1:', f1, 'p', p, 'r', r)

Evaluate: 100%|██████████| 216/216 [04:15<00:00,  1.18s/it]

CM:
 [[ 1146   635   944]
 [  295 13391  1231]
 [  318  1055  8515]]
f1: 0.8315952080368716 p 0.8317075652907133 r 0.8373410824555031


In [11]:
cm, f1, p, r = evaluate_baseline(
    test_loader=test_loader,
    model=model
)
print('CM:\n', cm)
print('f1:', f1, 'p', p, 'r', r)

Evaluate: 100%|██████████| 216/216 [01:51<00:00,  1.95it/s]

CM:
 [[ 1249   659   817]
 [  308 13246  1363]
 [  460   840  8588]]
f1: 0.8347409556236766 p 0.8345111204767544 r 0.8384671267707955


In [38]:
F1, P, R, all_extras, all_words_spacy = evaluate_closure(
    test_loader=test_loader,
    model=model,
    id2label=id2label       # Dict: {int_label: str_label}
)

Evaluating (closure): 100%|██████████| 216/216 [04:19<00:00,  1.20s/it]


Using closure-based evaluation for I2B2 (document-level).
====== [I2B2 CLOSURE-BASED EVALUATION] ======
Precision:       0.8381
Recall:          0.8412
F1-score:        0.8397
n_gold:          27530
n_pred:          27530
n_gold_closure:  175550
n_pred_closure:  127194
Evaluation finished in 260.44s.
